# Orca Nano — QLoRA Fine-Tune (Kaggle, free T4/P100)

**Kaggle setup before running any cell — different from Colab:**
1. **Enable GPU**: right sidebar → Settings → Accelerator → **GPU T4 x2** (or P100).
2. **Enable internet**: right sidebar → Settings → Internet → **On** (needed for pip install and downloading the base model — off by default on Kaggle).
3. **Upload the training data as a Kaggle Dataset first** (Kaggle has no `files.upload()` widget like Colab):
   - Go to kaggle.com/datasets → **New Dataset** → upload `orca_llama3_train.jsonl` and `orca_llama3_eval.jsonl` from your Desktop → create it (private is fine).
   - Back in this notebook: right sidebar → **Add Input** → search for the dataset you just created → add it.
   - Don't worry about the exact folder path — the next cell finds the files automatically under `/kaggle/input/`.

**Output**: unlike Colab, anything saved to `/kaggle/working/` automatically shows up in this notebook's **Output** tab after it finishes running — download directly from there, no Drive/download-link dance needed.

**Same training config as the Colab version** — rank 16 LoRA, 2050/108 dataset (full nano distillation batch + 150 targeted `honesty_hedging` examples), 2 epochs (not 3 — v1's validation loss rose after step 100, a real overfitting signal on this dataset size).

In [ ]:
# Disable the Xet fast-download path up front — this stalled a model download
# on Colab last round ("Download stalled for 'unsloth/qwen2.5-7b...'"). Same
# Hugging Face backend, same fix applies here.
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl transformers datasets peft bitsandbytes accelerate

## Find your uploaded training data

Searches recursively under `/kaggle/input/` so it doesn't matter what your
dataset was named or how Kaggle nested the folder.

In [ ]:
import glob

train_matches = glob.glob('/kaggle/input/**/orca_llama3_train.jsonl', recursive=True)
eval_matches  = glob.glob('/kaggle/input/**/orca_llama3_eval.jsonl', recursive=True)

print('Train file found:', train_matches)
print('Eval file found:', eval_matches)

if not train_matches:
    raise FileNotFoundError(
        "orca_llama3_train.jsonl not found under /kaggle/input/. "
        "Make sure you uploaded it as a Kaggle Dataset and added it via "
        "'Add Input' in the right sidebar."
    )

train_path = train_matches[0]
eval_path = eval_matches[0] if eval_matches else None

In [ ]:
import json

def load_jsonl(path):
    lines = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    lines.append(json.loads(line))
                except Exception:
                    pass
    return lines

raw_train = load_jsonl(train_path)
raw_eval  = load_jsonl(eval_path) if eval_path else raw_train[:max(1, len(raw_train)//10)]
print(f'train={len(raw_train)} eval={len(raw_eval)}')

## Load base model (4-bit) + attach LoRA

Rank 16 — sized for a free-tier GPU's VRAM, not the 128-rank A100 cloud preset.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
base_model = "unsloth/Qwen2.5-7B-Instruct"  # exact case matters on Hugging Face

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import Dataset

def format_conv(ex):
    turns = ex.get("conversations", ex.get("text"))
    if isinstance(turns, str):
        return turns  # already-formatted llama3 text field
    parts = []
    for t in turns:
        role = t.get("role", "")
        val  = t.get("value", "")
        if role == "system":
            parts.append(f"<|start_header_id|>system<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "human":
            parts.append(f"<|start_header_id|>user<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "gpt":
            parts.append(f"<|start_header_id|>assistant<|end_header_id|>\n\n{val}<|eot_id|>")
    return "".join(parts)

# The formatter.py output already has a 'text' field per example (llama3 format) — use it directly if present.
train_ds = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_train])
eval_ds  = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_eval])
print(f'train_ds={len(train_ds)} eval_ds={len(eval_ds)}')

## Train

Batch size 2 + grad accumulation 4 (effective batch 8), 2 epochs.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_steps=100,
        save_strategy="no",
        output_dir="/kaggle/working/output",
        eval_strategy="steps",
        report_to="none",
    ),
)

print("[train] starting QLoRA training...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[train] done in {elapsed:.1f} min")

## Merge LoRA + export GGUF to /kaggle/working/

Anything under `/kaggle/working/` auto-appears in the notebook's **Output**
tab once this cell finishes — download the `.gguf` file directly from there,
no Drive step needed.

In [ ]:
print("[merge] merging LoRA adapters...")
model.save_pretrained_merged("/kaggle/working/merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to /kaggle/working/merged")

print("[gguf] converting to GGUF q4_k_m...")
model.save_pretrained_gguf("/kaggle/working/gguf", tokenizer, quantization_method="q4_k_m")
print("[gguf] saved to /kaggle/working/gguf")

In [ ]:
import glob

candidates = [f for f in glob.glob('/kaggle/working/**/*.gguf', recursive=True) if 'q4_k_m' in f.lower()]
print('Found GGUF file(s):', candidates)
if candidates:
    print(f"\nGo to this notebook's 'Output' tab (or 'Data' > 'Output' in the\n"
          f"right sidebar) once this finishes running — {candidates[0].split('/')[-1]}\n"
          f"will be listed there for direct download.")
else:
    print('No GGUF file found — check the [gguf] step above for errors.')